<a href="https://colab.research.google.com/github/ttobias28/PhishRadar-Project/blob/main/tfidf_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhishRadar — Approach 1: TF-IDF + Logistic Regression for phishing detection

**Authors:** Connor Buttrey, Oliver Conley, Robert Morriss, Ashley Smutnik, Teagan Tobias

<br>

**Data: the team's cross-source splits** (`phishradar_splits.zip`). The model trains on **Enron, Ling and SpamAssassin** only. It is then tested on:

| Test set | What it tells you |
|---|---|
| `test_insource` | Same sources as training. The "normal" score. |
| `test_ceas` | A source it never saw (spam + legit). Does it generalize? |
| `test_nazario_paired` | Real phishing (Nazario) mixed 50/50 with unseen legit emails (CEAS). The closest thing to a real phishing test. |
| `test_nazario` | Real phishing only. Shows the detection rate (recall). |
| `test_nigerian` | Advance-fee scams only. Shows the detection rate (recall). |

**Every time (Google Colab):**

1. Open this notebook in Colab (colab.research.google.com → **File → Upload notebook**).
2. No GPU is needed for this model. The default runtime is fine.
3. Make sure the split CSVs are in your Google Drive at `MyDrive/phishradar/data/splits/`.
4. Run the cells top to bottom with **Shift+Enter** (or **Runtime → Run all**). The first cell asks to connect to Google Drive: click **Connect** and approve.
5. Results are saved to `MyDrive/phishradar/results/` in your Drive, so they are still there after the session ends.

**Running in plain Jupyter instead?** Put the CSVs in `data/splits/` next to this notebook. The notebook detects that it isn't in Colab and skips the Google Drive step.

**Run time:** about a minute. Training uses all ~30,000 training emails.

## 1. Setup: connect to your files

In [12]:
from pathlib import Path
import sklearn; print('scikit-learn', sklearn.__version__)

try:  # running in Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/phishradar')
except ImportError:  # running locally (VS Code / Jupyter): use the current folder
    BASE = Path('.')

SPLITS = BASE / 'data' / 'splits'
OUT = BASE / 'results' / 'predictions'
OUT.mkdir(parents=True, exist_ok=True)
print('Reading data from:', SPLITS.resolve())
print('Files found:', sorted(p.name for p in SPLITS.glob('*.csv')))

scikit-learn 1.6.1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading data from: /content/drive/MyDrive/phishradar/data/splits
Files found: ['test_ceas.csv', 'test_insource.csv', 'test_nazario.csv', 'test_nazario_paired.csv', 'test_nigerian.csv', 'train.csv', 'val.csv']


## 2. Load the data
The model only learns from `train.csv`. `val.csv` is for choosing settings. The `test_*` files are only used at the end.

In [13]:
import numpy as np
import pandas as pd

SEED = 42
TEST_FILES = ['test_insource', 'test_ceas', 'test_nazario', 'test_nigerian', 'test_nazario_paired']

train = pd.read_csv(SPLITS / 'train.csv')
val = pd.read_csv(SPLITS / 'val.csv')
print(len(train), 'train emails,', len(val), 'val emails')
train.head()

30217 train emails, 3778 val emails


,text,label,source
0,cheap oem software shipping worldwide . worryi...,1,Enron
1,Re: Wifi query I think what you're looking at ...,0,SpamAssassin
2,nels-29 final announcement final announcement ...,0,Ling
3,"coca cola , mbna america , nascar partner with...",1,Enron
4,hi 325 and hi 368 revision effective 6 / 10 / ...,0,Enron


## 3. Turn emails into numbers (TF-IDF)
The vectorizer learns its vocabulary from **training text only**. It never sees val or test text.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=100_000)
X_train = vec.fit_transform(train['text'])
X_val = vec.transform(val['text'])
print('vocabulary size:', len(vec.vocabulary_))

vocabulary size: 100000


## 4. Train, trying a few settings
`C` controls how strongly the model fits the training data. We keep whichever `C` scores best on the validation set.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

best = None
for C in [0.1, 1, 10, 100]:
    model = LogisticRegression(C=C, max_iter=2000, random_state=SEED)
    model.fit(X_train, train['label'])
    f1 = f1_score(val['label'], model.predict(X_val))
    print(f'C={C:<6} validation F1 = {f1:.4f}')
    if best is None or f1 > best[0]:
        best = (f1, C, model)

_, best_C, model = best
print('Best C:', best_C)

C=0.1    validation F1 = 0.9766
C=1      validation F1 = 0.9861
C=10     validation F1 = 0.9886
C=100    validation F1 = 0.9880
Best C: 10


## 5. Predict every test file and save the predictions
Each file saved is in the team format: `row_id, y_true, y_pred, score`.

In [16]:
from sklearn.metrics import precision_score, recall_score

def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    has_legit = (y == 0).any()   # Nazario / Nigerian have no legitimate emails
    return {
        'rows': len(y),
        'recall': recall_score(y, p, zero_division=0),
        'precision': precision_score(y, p, zero_division=0) if has_legit else np.nan,
        'f1': f1_score(y, p, zero_division=0) if has_legit else np.nan,
        'fpr': ((p == 1) & (y == 0)).sum() / (y == 0).sum() if has_legit else np.nan,
    }

rows = []
for name in ['val'] + TEST_FILES:
    df = val if name == 'val' else pd.read_csv(SPLITS / f'{name}.csv')
    X = X_val if name == 'val' else vec.transform(df['text'])
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    if name != 'val':
        pd.DataFrame({'row_id': range(len(df)), 'y_true': df['label'],
                      'y_pred': preds, 'score': probs}
                     ).to_csv(OUT / f'tfidf_{name}.csv', index=False)
    rows.append({'set': name, **metrics(df['label'], preds)})

results = pd.DataFrame(rows).round(4)
results.to_csv(BASE / 'results' / 'tfidf_results.csv', index=False)
results

,set,rows,recall,precision,f1,fpr
0,val,3778,0.9880,0.9893,0.9886,0.0078
1,test_insource,3778,0.9912,0.9887,0.9899,0.0082
2,test_ceas,38748,0.8500,0.9218,0.8845,0.0896
3,test_nazario,1544,0.7902,NaN,NaN,NaN
4,test_nigerian,3243,0.9935,NaN,NaN,NaN
5,test_nazario_paired,3088,0.7902,0.8931,0.8385,0.0946


**Reading the table:** `recall` = share of real phishing/scam caught. `precision` = share of flagged emails that were really phishing. `fpr` = share of legitimate emails wrongly flagged. Nazario and Nigerian have no legitimate emails, so only recall is meaningful there (shown as NaN for the rest).

## 6. Which words does the model rely on?

In [17]:
words = np.array(vec.get_feature_names_out())
order = np.argsort(model.coef_[0])
top = pd.concat([
    pd.DataFrame({'direction': 'phishing', 'word': words[order[::-1][:30]], 'weight': model.coef_[0][order[::-1][:30]]}),
    pd.DataFrame({'direction': 'legitimate', 'word': words[order[:30]], 'weight': model.coef_[0][order[:30]]}),
])
top.to_csv(BASE / 'results' / 'tfidf_top_words.csv', index=False)
print('Top phishing words:  ', ', '.join(top[top.direction == 'phishing'].word.head(15)))
print('Top legitimate words:', ', '.join(top[top.direction == 'legitimate'].word.head(15)))

Top phishing words:   your, 2004, 2005, remove, you, money, site, our, here, mobile, no, hello, life, viagra, software
Top legitimate words: enron, vince, thanks, 2002, the, louise, attached, 2001, wrote, 713, 2000, language, doc, gas, edu


**Look at these words.** If you see names like `enron` or years like `2004`, the model is partly recognizing *which dataset* an email came from, not just phishing. Worth discussing in the report.